# CLEANS ABSTRAK PRODI MANAJEMEN

In [1]:
!pip install plotly
!pip install --upgrade gensim

In [2]:
from gensim.models import Word2Vec, FastText
import pandas as pd
import re

from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer

from matplotlib import pyplot as plt
import plotly.graph_objects as go

import numpy as np

import warnings
warnings.filterwarnings('ignore')

In [3]:


# Load file CSV kamu
df = pd.read_csv("/content/preprocessing_result.csv")

# Cek 5 data pertama
print(df.head())

   id                                              title  \
0   0  PENGARUH FAKTOR-FAKTOR PELATIHAN DAN PENGEMBAN...   
1   1  ANALISIS PERSEPSI BRAND ASSOCIATION MENURUT PE...   
2   2  PENGARUH GAYA KEPEMIMPINAN DEMOKRATIK TERHADAP...   
3   3  Pengukuran Website Quality Pada Situs Sistem A...   
4   4  PENGARUH KEPEMIMPINAN DAN KOMPENSASI TERHADAP ...   

                    author                                       abstract_ind  \
0                  SATIYAH  si\n\n\n\n\n                                  ...   
1                  Faishal  si\n\n\n\n\nTujuan penelitian ini adalah untuk...   
2          Wahyu Kurniawan                                                 si   
3   Muhammad Zakaria Utomo  si\n\n\n\n\nAplikasi nyata pemanfaatan teknolo...   
4  Hendri Wahyudi Prayitno  si\n\n\n\n\nAbstrak\r\nPenelitian ini mengguna...   

                                                 url  \
0  https://pta.trunojoyo.ac.id/welcome/detail/080...   
1  https://pta.trunojoyo.ac.id/welcome/d

## TF-IDF

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Load file hasil preprocessing
df = pd.read_csv("/content/preprocessing_result.csv")

# Ubah kolom tokens dari string ke list lalu gabungkan jadi string
df["tokens_list"] = df["tokens"].apply(eval)
df["tokens_str"] = df["tokens_list"].apply(lambda x: " ".join(x))

# Buat TF-IDF
vectorizer = TfidfVectorizer()
X_tfidf = vectorizer.fit_transform(df["tokens_str"])

# Ubah jadi DataFrame biar rapi
tfidf_df = pd.DataFrame(
    X_tfidf.toarray(),
    columns=vectorizer.get_feature_names_out()
)

# Tambahkan ID dokumen biar jelas
tfidf_df.insert(0, "doc_id", df["id"])

# Info shape
print("TF-IDF shape:", X_tfidf.shape)

# Tampilkan 5 baris pertama
print(tfidf_df.head())

TF-IDF shape: (1000, 5632)
   doc_id  aaa  aaaamanahsyariah  aar  abadi  abai  abalisis  abc  abcs  abd  \
0       0  0.0               0.0  0.0    0.0   0.0       0.0  0.0   0.0  0.0   
1       1  0.0               0.0  0.0    0.0   0.0       0.0  0.0   0.0  0.0   
2       2  0.0               0.0  0.0    0.0   0.0       0.0  0.0   0.0  0.0   
3       3  0.0               0.0  0.0    0.0   0.0       0.0  0.0   0.0  0.0   
4       4  0.0               0.0  0.0    0.0   0.0       0.0  0.0   0.0  0.0   

   ...  zscore  zte  zuhri  zuhruf  zulfi  zulia  zuliana  zulkifli  zulpah  \
0  ...     0.0  0.0    0.0     0.0    0.0    0.0      0.0       0.0     0.0   
1  ...     0.0  0.0    0.0     0.0    0.0    0.0      0.0       0.0     0.0   
2  ...     0.0  0.0    0.0     0.0    0.0    0.0      0.0       0.0     0.0   
3  ...     0.0  0.0    0.0     0.0    0.0    0.0      0.0       0.0     0.0   
4  ...     0.0  0.0    0.0     0.0    0.0    0.0      0.0       0.0     0.0   

   zyn  
0  0.0  

## WORD EMBEDDING

In [13]:
from gensim.models import Word2Vec

# Convert tokens jadi list of list
token_lists = df["tokens"].apply(eval).tolist()

# Train Word2Vec
w2v_model = Word2Vec(sentences=token_lists, vector_size=100, window=5, min_count=1, workers=4)

# Representasi dokumen = rata-rata vektor kata di abstrak
import numpy as np
def doc_vector(tokens):
    vecs = [w2v_model.wv[word] for word in tokens if word in w2v_model.wv]
    return np.mean(vecs, axis=0) if len(vecs) > 0 else np.zeros(w2v_model.vector_size)

X_w2v = np.array([doc_vector(tokens) for tokens in token_lists])

print("Shape Word2Vec:", X_w2v.shape)


Shape Word2Vec: (1000, 100)


In [15]:
# Import library
import pandas as pd
from gensim.models import Word2Vec

# Load hasil preprocessing
df = pd.read_csv("/content/preprocessing_result.csv")

# Kolom "tokens" isinya list dalam bentuk string, ubah ke list beneran
df["tokens_list"] = df["tokens"].apply(eval)

# Buat corpus (list of list of tokens)
corpus = df["tokens_list"].tolist()

# Tampilkan contoh corpus pertama
print("Contoh corpus dokumen 0:\n", corpus[0])

# Latih Word2Vec
model = Word2Vec(
    sentences=corpus,
    vector_size=56,   # bisa diganti sesuai kebutuhan
    window=5,
    min_count=1,
    sg=1              # 1 = skip-gram, 0 = CBOW
)

# ==== Eksplorasi Embeddings ====

# Function to safely get most similar words
def safe_most_similar(model, word, topn=5):
    if word in model.wv:
        print(f"\nKata mirip dengan '{word}':")
        print(model.wv.most_similar(word, topn=topn))
    else:
        print(f"\nKata '{word}' not found in vocabulary.")

# kata mirip dengan "penelitian"
safe_most_similar(model, 'penelitian')

# kata mirip dengan "data"
safe_most_similar(model, 'data')

# cosmul example - check if all positive and negative words are in vocab
positive_words = ['penelitian', 'sistem']
negative_words = ['data']
if all(word in model.wv for word in positive_words + negative_words):
    print(f"\nCosmul ({' + '.join(positive_words)} - {' + '.join(negative_words)}):")
    print(model.wv.most_similar_cosmul(
        positive=positive_words,
        negative=negative_words,
        topn=5
    ))
else:
    missing_words = [word for word in positive_words + negative_words if word not in model.wv]
    print(f"\nCannot perform Cosmul. Missing words in vocabulary: {', '.join(missing_words)}")


# doesnt_match
match_words = "penelitian data sistem informasi".split()
if all(word in model.wv for word in match_words):
    print(f"\nKata yang tidak cocok dalam {match_words}:")
    print(model.wv.doesnt_match(match_words))
else:
     missing_words = [word for word in match_words if word not in model.wv]
     print(f"\nCannot perform doesnt_match. Missing words in vocabulary: {', '.join(missing_words)}")


# ==== Save Embeddings ====
filename = "pta_embeddings.txt"
model.wv.save_word2vec_format(filename, binary=False)
print(f"\nEmbeddings disimpan ke {filename}")

Contoh corpus dokumen 0:
 ['abstrak', 'satiyah', 'pengaruh', 'faktor', 'faktor', 'latih', 'kembang', 'produktivitas', 'kerja', 'dinas', 'laut', 'ikan', 'bangkal', 'bawah', 'bimbing', 'dra', 'anugrahini', 'irawati', 'helm', 'buyung', 'aulia', 'upaya', 'tingkat', 'produktivitas', 'kerja', 'mudah', 'salah', 'usaha', 'produktivitas', 'tingkat', 'terap', 'program', 'latih', 'kembang', 'sumber', 'daya', 'manusia', 'sdm', 'laksana', 'instansi', 'produktivitas', 'capai', 'tingkat', 'mampu', 'pegawai', 'efektif', 'efisien', 'latih', 'pengembnagan', 'harap', 'pegawai', 'sesuai', 'butuh', 'butuh', 'sikap', 'tingkah', 'laku', 'terampil', 'tahu', 'sesuai', 'tuntut', 'ubah', 'latih', 'kembang', 'pegawai', 'dukung', 'cipta', 'suasana', 'kerja', 'kondusif', 'instansi', 'produktivitas', 'kerja', 'tingkat', 'tuju', 'teliti', 'pengaruh', 'faktor', 'faktor', 'latih', 'kembang', 'produktivitas', 'kerja', 'dinas', 'laut', 'ikan', 'bangkal', 'ukur', 'menganalisa', 'hubung', 'variabel', 'teliti', 'dekat', 'ob

In [16]:
class MyTokenizer:
    def fit_transform(self, texts):
        # Tokenisasi sederhana: lowercase + split
        return [str(text).lower().split() for text in texts]

class MeanEmbeddingVectorizer:
    def __init__(self, word2vec_model):
        self.word2vec = word2vec_model
        # Perbaikan: gunakan vector_size (Gensim ≥ 4.0)
        self.dim = word2vec_model.wv.vector_size

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_tokenized = MyTokenizer().fit_transform(X)
        embeddings = []
        for words in X_tokenized:
            # Ambil vektor hanya untuk kata yang ada di vocab
            valid_vectors = [
                self.word2vec.wv[word] for word in words
                if word in self.word2vec.wv
            ]
            if valid_vectors:
                embeddings.append(np.mean(valid_vectors, axis=0))
            else:
                embeddings.append(np.zeros(self.dim))
        return np.array(embeddings)

    def fit_transform(self, X, y=None):
        return self.transform(X)

In [18]:
df.shape

(1000, 7)

In [20]:
mean_embedding_vectorizer = MeanEmbeddingVectorizer(model)
mean_embedded = mean_embedding_vectorizer.fit_transform(df['tokens_list'])

In [21]:
df['array']=list(mean_embedded)

In [22]:
df.head(5)

,id,title,author,abstract_ind,url,tokens,tokens_list,array
0,0,PENGARUH FAKTOR-FAKTOR PELATIHAN DAN PENGEMBAN...,SATIYAH,si\n\n\n\n\n ...,https://pta.trunojoyo.ac.id/welcome/detail/080...,"['abstrak', 'satiyah', 'pengaruh', 'faktor', '...","[abstrak, satiyah, pengaruh, faktor, faktor, l...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,1,ANALISIS PERSEPSI BRAND ASSOCIATION MENURUT PE...,Faishal,si\n\n\n\n\nTujuan penelitian ini adalah untuk...,https://pta.trunojoyo.ac.id/welcome/detail/090...,"['tuju', 'teliti', 'persepsi', 'brand', 'assoc...","[tuju, teliti, persepsi, brand, association, l...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,2,PENGARUH GAYA KEPEMIMPINAN DEMOKRATIK TERHADAP...,Wahyu Kurniawan,si,https://pta.trunojoyo.ac.id/welcome/detail/080...,[],[],"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,3,Pengukuran Website Quality Pada Situs Sistem A...,Muhammad Zakaria Utomo,si\n\n\n\n\nAplikasi nyata pemanfaatan teknolo...,https://pta.trunojoyo.ac.id/welcome/detail/100...,"['aplikasi', 'nyata', 'manfaat', 'teknologi', ...","[aplikasi, nyata, manfaat, teknologi, informas...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,4,PENGARUH KEPEMIMPINAN DAN KOMPENSASI TERHADAP ...,Hendri Wahyudi Prayitno,si\n\n\n\n\nAbstrak\r\nPenelitian ini mengguna...,https://pta.trunojoyo.ac.id/welcome/detail/080...,"['abstrak', 'teliti', 'metode', 'kuantitatif',...","[abstrak, teliti, metode, kuantitatif, tekan, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [24]:
df['embedding_length'] = df['array'].str.len()


In [25]:
print(df['embedding_length'])

0      56
1      56
2      56
3      56
4      56
       ..
995    56
996    56
997    56
998    56
999    56
Name: embedding_length, Length: 1000, dtype: int64


In [26]:
df.shape

(1000, 9)

In [31]:
import pandas as pd
import numpy as np
from gensim.models import Word2Vec

# 1. Load dataset
df = pd.read_csv("preprocessing_result.csv")

# 2. Pastikan kolom tokens ada (string daftar kata)
df["tokens_str"] = df["tokens"].apply(lambda x: " ".join(eval(x)) if isinstance(x, str) else "")

# 3. Bikin corpus untuk Word2Vec
corpus = [text.split() for text in df["tokens_str"]]

# 4. Latih Word2Vec
model = Word2Vec(corpus, vector_size=56, min_count=1, workers=4)

# 5. MeanEmbeddingVectorizer
class MyTokenizer:
    def fit_transform(self, texts):
        return [str(text).lower().split() for text in texts]

class MeanEmbeddingVectorizer:
    def __init__(self, word2vec_model):
        self.word2vec = word2vec_model
        self.dim = word2vec_model.wv.vector_size

    def transform(self, X):
        X_tokenized = MyTokenizer().fit_transform(X)
        embeddings = []
        for words in X_tokenized:
            valid_vectors = [self.word2vec.wv[w] for w in words if w in self.word2vec.wv]
            if valid_vectors:
                embeddings.append(np.mean(valid_vectors, axis=0))
            else:
                embeddings.append(np.zeros(self.dim))
        return np.array(embeddings)

# 6. Transform teks jadi embedding
texts = df["tokens_str"].tolist()
vectorizer = MeanEmbeddingVectorizer(model)
X_embed = vectorizer.transform(texts)

print("Shape embeddings:", X_embed.shape)
print("Contoh dokumen pertama:\n", X_embed[0])

# 7. Simpan ke DataFrame
num_features = X_embed.shape[1]
columns = [f"f{i+1}" for i in range(num_features)]
embedding_df = pd.DataFrame(X_embed, columns=columns)

# Gabung metadata
final_df = pd.concat([df[["id","title","author","url"]], embedding_df], axis=1)

# 8. Save ke CSV
final_df.to_csv("word_embeddings_features.csv", index=False)
print("\nHasil embeddings disimpan ke 'word_embeddings_features.csv'")


Shape embeddings: (1000, 56)
Contoh dokumen pertama:
 [ 0.41714409 -0.04168329 -0.31555137  0.44645479  0.50892955 -0.68637031
 -0.00145381 -0.55331463 -0.75308633 -0.21033765 -0.42613235  0.35601047
 -0.84386533 -0.54033089  0.33936727  0.52722842  0.12394693  0.17974444
 -0.02824138 -0.43485069  0.68457896 -0.01475596  0.63503301 -0.66723561
  0.29311672 -0.32292941  1.12548912  0.02216942  0.4895356  -0.22547728
 -0.11266289  0.50150508 -0.02858619  0.7652728  -0.51093131 -0.3075712
  0.34432188  0.10526028 -0.38155389  0.54580456 -0.80889434  0.04751804
  0.75979298  0.00217088  0.46299919 -0.24621116  0.00330278  0.25611392
  0.16526414  0.44888785 -1.06781316  0.28689393 -0.25207508  0.31879416
 -0.26872405 -0.5188995 ]

Hasil embeddings disimpan ke 'word_embeddings_features.csv'


In [34]:
# Lihat 5 baris pertama
embedding_df.head()

# Lihat ukuran DataFrame
print("Shape:", embedding_df.shape)

# Lihat ringkasan tiap kolom
print(embedding_df.info())


Shape: (1000, 56)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 56 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   f1      1000 non-null   float64
 1   f2      1000 non-null   float64
 2   f3      1000 non-null   float64
 3   f4      1000 non-null   float64
 4   f5      1000 non-null   float64
 5   f6      1000 non-null   float64
 6   f7      1000 non-null   float64
 7   f8      1000 non-null   float64
 8   f9      1000 non-null   float64
 9   f10     1000 non-null   float64
 10  f11     1000 non-null   float64
 11  f12     1000 non-null   float64
 12  f13     1000 non-null   float64
 13  f14     1000 non-null   float64
 14  f15     1000 non-null   float64
 15  f16     1000 non-null   float64
 16  f17     1000 non-null   float64
 17  f18     1000 non-null   float64
 18  f19     1000 non-null   float64
 19  f20     1000 non-null   float64
 20  f21     1000 non-null   float64
 21  f22     1000 non-nul

In [36]:
num_features = X_embed.shape[1]
columns = [f"f{i+1}" for i in range(num_features)]
embedding_df = pd.DataFrame(X_embed, columns=columns)


In [37]:
embedding_df

,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,...,f47,f48,f49,f50,f51,f52,f53,f54,f55,f56
0,0.417144,-0.041683,-0.315551,0.446455,0.508930,-0.686370,-0.001454,-0.553315,-0.753086,-0.210338,...,0.003303,0.256114,0.165264,0.448888,-1.067813,0.286894,-0.252075,0.318794,-0.268724,-0.518900
1,-0.137392,0.165126,-0.100648,-0.069360,0.200711,-0.847127,0.336629,-0.370019,-0.038729,-0.208057,...,-0.290751,0.141591,-0.095891,0.294693,-0.571569,-0.275432,-0.081239,0.175270,-0.489611,-0.272008
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.114640,0.055883,-0.177440,0.095860,0.152385,-0.693575,0.113620,-0.349148,-0.316113,-0.098577,...,-0.282278,0.170123,-0.071743,0.298444,-0.681455,-0.042836,-0.286440,0.208620,-0.278755,-0.362300
4,0.683831,-0.266939,-0.856874,0.506953,0.864336,-0.937824,0.344530,-0.452965,-0.843173,-0.578207,...,0.356841,0.113071,-0.059922,0.958993,-0.999742,-0.019101,-0.343978,-0.003225,-0.129976,-0.519661
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,0.782306,-0.206646,-0.705584,0.614447,0.788802,-0.862621,0.158578,-0.544823,-1.045305,-0.439714,...,0.375423,0.290584,0.172655,0.892652,-1.207033,0.193182,-0.370929,0.147630,-0.056323,-0.551012
996,-0.130322,0.584023,0.025805,-0.024237,0.212774,-0.764588,0.200262,-0.565330,-0.387611,-0.418711,...,0.329708,0.617355,0.327932,0.640310,-0.534373,-0.248987,0.117682,0.013998,-0.175723,-0.280828
997,0.713866,-0.098278,-0.479808,0.562241,0.622332,-0.676903,-0.024263,-0.576127,-1.034340,-0.390121,...,0.391619,0.428113,0.345405,0.762416,-1.177234,0.324904,-0.340540,0.295565,-0.091730,-0.553174
998,0.220254,0.108196,-0.309945,0.231045,0.380492,-0.718658,0.227308,-0.402759,-0.590636,-0.195823,...,0.066948,0.276584,0.053513,0.507536,-0.821637,-0.040880,-0.158862,0.249737,-0.214832,-0.372810


In [38]:
embedding_df.shape

(1000, 56)

# BERITA CLEANS